### Connect postgresql database

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text

# 数据库配置
username = "XXXXXX"
password = "YYYYYY"
host = "localhost"
port = 5432
database = "eyewear-data"

# 创建连接
engine = create_engine(
    f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}"
)


### Query and cal customer RFM feature

In [ ]:
# 1 查询并计算客户的RFM特征
customer_rfm_feature_sql = """
WITH customer_orders AS (
    SELECT 
        c.customer_id,
        c.total_orders,
        c.total_spend,
        c.total_spend / NULLIF(c.total_orders,0) AS avg_order_value,
        MAX(o.order_date) AS last_purchase_date,
        MIN(o.order_date) AS first_purchase_date,
        -- 是否购买过
        CASE WHEN COUNT(o.order_id) > 0 THEN 1 
        ELSE 0 END AS has_purchase,
        -- 是否复购客户
        CASE WHEN COUNT(o.order_id) >= 2 THEN 1 ELSE 0
        END AS repeat_customer_flag,
        CASE WHEN c.registration_date IS NOT NULL THEN COUNT(o.order_id)::float /
        NULLIF(DATE '2025-06-01'-c.registration_date, 0) ELSE 0 END AS purchase_frequency,
        CASE WHEN c.total_orders > 1 THEN (c.total_orders - 1.0) / NULLIF(c.total_orders, 0) 
        ELSE 0 END AS repeat_purchase_rate -- 复购率
    
    FROM "CustomerInfo" c
    LEFT JOIN "Order" o
        ON c.customer_id=o.customer_id
    AND o.order_status IN ('Completed', 'Shipped')
    LEFT JOIN (
        SELECT 
            customer_id,
            order_date,
            order_date - LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date) AS order_interval_days
        FROM "Order"
    ) intervals 
        ON o.customer_id = intervals.customer_id 
        AND o.order_date = intervals.order_date
    WHERE o.order_status IN ('Completed','Shipped')
    OR o.order_id IS NULL          -- 保留没有订单的客户
    GROUP BY 
        c.customer_id,
        c.total_orders,
        c.total_spend,
        c.registration_date
)

SELECT 
    c.customer_id,
    c.registration_date,
    co.total_orders,
    co.total_spend,
    co.avg_order_value,
    co.first_purchase_date,
    co.last_purchase_date,
    co.purchase_frequency,
    co.repeat_customer_flag,
    co.has_purchase,
    co.repeat_purchase_rate,             -- 新增
    DATE '2025-06-01' - c.registration_date AS registration_days,
    DATE '2025-06-01' - c.last_purchase_date AS recency_days
FROM "CustomerInfo" c
LEFT JOIN customer_orders co ON c.customer_id = co.customer_id;
"""

df_rfm = pd.read_sql(customer_rfm_feature_sql, engine)

df_rfm

In [ ]:
df_rfm.columns

### Query and cal customer buying product feature

In [ ]:
# 2 查询并计算客户购买Product的特征
customer_product_feature_sql = """
SELECT
    c.customer_id,
    SUM(CASE WHEN oi.line_price_after_tax > 0 THEN oi.quantity ELSE 0 END) AS total_items,   
    -- 镜架类数量比例
    SUM(CASE WHEN  p.category IN ('Sunglasses','Eyeglasses','AI Glasses') AND oi.line_price_after_tax > 0 THEN oi.quantity ELSE 0 END)::float / NULLIF(SUM(CASE WHEN oi.line_price_after_tax > 0 THEN oi.quantity ELSE 0 END),0) AS frame_ratio,
    -- 镜片数量比例
    SUM(CASE WHEN p.category='Lens' AND oi.line_price_after_tax > 0 THEN oi.quantity ELSE 0 END)::float / NULLIF(SUM(CASE WHEN oi.line_price_after_tax > 0 THEN oi.quantity ELSE 0 END),0) AS lens_ratio,
    -- AI Glasses数量比例
    SUM(CASE WHEN p.category='AI Glasses' AND oi.line_price_after_tax > 0 THEN oi.quantity ELSE 0 END)::float / NULLIF(SUM(CASE WHEN oi.line_price_after_tax > 0 THEN oi.quantity ELSE 0 END),0) AS ai_glasses_ratio,
    -- Sunglasses数量比例
    SUM(CASE WHEN p.category='Sunglasses' AND oi.line_price_after_tax > 0 THEN oi.quantity ELSE 0 END)::float / NULLIF(SUM(CASE WHEN oi.line_price_after_tax > 0 THEN oi.quantity ELSE 0 END),0) AS sunglasses_ratio

FROM "CustomerInfo" c
LEFT JOIN "Order" o
ON c.customer_id=o.customer_id
LEFT JOIN "OrderItem" oi
ON o.order_id=oi.order_id
LEFT JOIN "ProductInfo" p
ON oi.product_id=p.product_id
WHERE o.order_status IN ('Completed','Shipped')
OR o.order_id IS NULL
GROUP BY c.customer_id;
"""

df_product = pd.read_sql(customer_product_feature_sql, engine)
df_product

### Query and cal customer attend promotion feature

In [ ]:
# 3 查询并计算客户参加的促销特征
customer_promotion_feature_sql = """
SELECT
    -- 平均折扣
    c.customer_id,
    -- 平均优惠比例
    1 - AVG(NULLIF(oi.discount_rate,0)) AS avg_discount_percentage,
    -- 使用优惠商品比例（排除赠品）
    AVG(CASE WHEN oi.is_free_gift=False AND oi.discount_rate < 1 THEN 1 ELSE 0 END) AS discount_usage_rate, 
    -- 促销订单比例
    COUNT(DISTINCT CASE WHEN o.campaign_id IS NOT NULL THEN o.order_id END)::float / NULLIF(COUNT(DISTINCT o.order_id), 0) AS promo_order_ratio

FROM "CustomerInfo" c
LEFT JOIN "Order" o
ON c.customer_id=o.customer_id
LEFT JOIN "OrderItem" oi
ON o.order_id=oi.order_id
WHERE o.order_status IN ('Completed','Shipped') OR o.order_id IS NULL
GROUP BY c.customer_id;
"""

df_promo = pd.read_sql(customer_promotion_feature_sql, engine)

df_promo

### Merge 3 dataframes into a complete dataframe

In [ ]:
df_all_customer_feature = (
    df_rfm
    .merge(df_product,
           on="customer_id",
           how="left")
    .merge(df_promo,
           on="customer_id",
           how="left")
)

df_has_purchase_customer = df_all_customer_feature[df_all_customer_feature.has_purchase==1].copy()
df_has_purchase_customer

In [ ]:
df_has_purchase_customer.shape
df_has_purchase_customer.columns

In [ ]:
df_has_purchase_customer[
    "avg_discount_percentage"
].isnull().sum()

In [ ]:
df_has_purchase_customer["avg_discount_percentage"].describe()

In [ ]:
# 处理目标列的缺失值
time_features = ["registration_days", "recency_days"]

normal_features = [
    "total_orders", "total_spend", "avg_order_value",
    "purchase_frequency", "repeat_customer_flag", "has_purchase",
    "repeat_purchase_rate",
    "total_items", "frame_ratio", "lens_ratio",
    "ai_glasses_ratio", "sunglasses_ratio",
    "avg_discount_percentage", "discount_usage_rate", "promo_order_ratio"
]

X_normal = df_has_purchase_customer[normal_features].copy()
X_normal = X_normal.fillna(0)

# # 2. recency_days 缺失值填最大值
X_time = df_has_purchase_customer[time_features].copy()
max_recency = X_time["recency_days"].max(skipna=True)
if pd.isna(max_recency):
    max_recency = 0
X_time["recency_days"] = X_time["recency_days"].fillna(max_recency)

for col in time_features:
    if pd.api.types.is_timedelta64_dtype(X_time[col]):
        X_time[col] = X_time[col].dt.total_seconds() / 86400

# 如果你后面只想对 normal_features 聚类，就用 X_normal
# 如果你想把 time_features 也一起纳入聚类，再合并：
X = pd.concat([X_normal, X_time], axis=1)
X



In [ ]:
X.isna().sum()

In [ ]:
X.dtypes

In [ ]:
# 归一化/标准化
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_scaled

In [ ]:
# PCA
# 在 X_scaled 生成之后立即插入
from sklearn.decomposition import PCA

# 先拟合完整的 PCA 看解释率
pca = PCA()
pca.fit(X_scaled)

# 查看每个主成分的解释率
explained_variance = pd.DataFrame({
    'PC': range(1, len(pca.explained_variance_ratio_) + 1),
    'Variance_Explained': pca.explained_variance_ratio_,
    'Cumulative_Variance': pca.explained_variance_ratio_.cumsum()
})
print(explained_variance.head(10))

# 画解释率曲线
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 5))
plt.plot(range(1, len(pca.explained_variance_ratio_) + 1), 
         pca.explained_variance_ratio_.cumsum(), marker='o')
plt.axhline(y=0.80, color='r', linestyle='--', label='80% explained')
plt.axhline(y=0.90, color='g', linestyle='--', label='90% explained')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA Explained Variance')
plt.legend()
plt.grid()
plt.show()


### Generate the final cluster label

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=5, random_state=42) # K=5的情况
df_has_purchase_customer["cluster"] = kmeans.fit_predict(X_scaled)
df_has_purchase_customer

### export customer cluster data

In [ ]:
df_has_purchase_customer.to_parquet(
    "customerinfo_cluster.parquet",
    index=False
)

In [ ]:
# 每个cluster的大小
df_has_purchase_customer["cluster"].value_counts()

In [ ]:
len(df_has_purchase_customer), X_scaled.shape[0]

### cluster stability test

In [ ]:
from sklearn.metrics import adjusted_rand_score

labels_ref = kmeans.labels_

aris = []
for seed in [0, 7, 42, 99, 123]:
    km = KMeans(n_clusters=5, random_state=seed, n_init=20)
    labels = km.fit_predict(X_scaled)
    aris.append(adjusted_rand_score(labels_ref, labels))

print("ARI:", aris)
print("平均 ARI:", sum(aris) / len(aris))

In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

base_labels = df_has_purchase_customer["cluster"].values
ari_scores = []

for i in range(10):
    # 随机抽80%客户
    sample_idx = np.random.choice(
        len(X_scaled),
        size=int(len(X_scaled)*0.8),
        replace=False
    )
    X_sample = X_scaled[sample_idx]
    kmeans_boot = KMeans(
        n_clusters=5,
        random_state=42,
        n_init=20
    )
    sample_labels = kmeans_boot.fit_predict(
        X_sample
    )
    # 和原始聚类比较
    ari = adjusted_rand_score(
        base_labels[sample_idx],
        sample_labels
    )
    ari_scores.append(ari)

print("Mean ARI:", np.mean(ari_scores))
print("Std ARI:", np.std(ari_scores))

### Determining the best Number of Clusters k

In [ ]:
# 确定最佳簇数 k
from sklearn.metrics import silhouette_score

inertia = []
silhouette = []
K = range(2, 9)

for k in K:
    km = KMeans(n_clusters=k, random_state=42,n_init="auto")
    labels = km.fit_predict(X_scaled)
    inertia.append(km.inertia_)
    silhouette.append(silhouette_score(X_scaled, labels,sample_size=10000, random_state=42))

result = pd.DataFrame({"k": list(K), "inertia": inertia, "silhouette": silhouette})
result

In [ ]:
# 生成 cluster_profile

cluster_profile = (
    df_has_purchase_customer
    .groupby("cluster")
    [
        [
            "total_orders",
            "total_spend",
            "avg_order_value",
            "purchase_frequency",
            "recency_days",
            "total_items",
            "frame_ratio",
            "lens_ratio",
            "ai_glasses_ratio",
            "sunglasses_ratio",
            "avg_discount_percentage",
            "promo_order_ratio"
        ]
    ]
    .mean()
)

cluster_profile

In [ ]:
# 数据可视化
# 降到 2D 用于可视化
pca_2d = PCA(n_components=2)
X_pca = pca_2d.fit_transform(X_scaled)

# 画PCA散点图
plt.figure(figsize=(10, 7))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], 
                      c=df_has_purchase_customer['cluster'], 
                      cmap='viridis', alpha=0.6, s=20)
plt.colorbar(scatter, label='Cluster')
plt.xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.2%} var)')
plt.ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.2%} var)')
plt.title('Customer Clusters in 2D PCA Space')
plt.grid(alpha=0.3)
plt.show()

print(f"前2个主成分共解释 {pca_2d.explained_variance_ratio_.sum():.2%} 的方差")

In [ ]:
# 统计每个cluster业务指标（版本1）

cluster_summary = (
    df_has_purchase_customer
    .groupby("cluster")
    .agg(
        customer_count=("customer_id", "count"),
        total_spend=("total_spend", "sum"),
        avg_spend=("total_spend", "mean"),
        avg_order_value=("avg_order_value", "mean"),
        avg_purchase_frequency=("purchase_frequency", "mean"),
        avg_recency_days=("recency_days", "mean"),
        avg_repeat_rate=("repeat_purchase_rate", "mean"),
        avg_discount_percentage=("avg_discount_percentage", "mean"),
        avg_promo_order_ratio=("promo_order_ratio", "mean"),
        avg_total_items=("total_items", "mean"),
        avg_frame_ratio=("frame_ratio", "mean"),
        avg_lens_ratio=("lens_ratio", "mean"),
        avg_ai_glasses_ratio=("ai_glasses_ratio", "mean"),
        avg_sunglasses_ratio=("sunglasses_ratio", "mean"),
    )
    .reset_index()
)

cluster_summary["customer_share"] = (
    cluster_summary.customer_count /
    cluster_summary.customer_count.sum()
)


cluster_summary["revenue_share"] = (
    cluster_summary.total_spend /
    cluster_summary.total_spend.sum()
)

cluster_summary

### data vis 1

In [ ]:
# 数据可视化还可以做：Radar map / heat map

import seaborn as sns
import matplotlib.pyplot as plt

# 如果你用的是 cluster_profile
profile = cluster_profile.copy()

plt.figure(figsize=(12, 6))
sns.heatmap(
    profile,
    annot=True,
    fmt=".2f",
    cmap="YlGnBu",
    linewidths=0.5
)
plt.title("Cluster Feature Profile Heatmap")
plt.xlabel("Feature")
plt.ylabel("Cluster")
plt.tight_layout()
plt.show()

### data vis 2

In [ ]:
profile_norm = (profile - profile.min()) / (profile.max() - profile.min())

plt.figure(figsize=(12, 6))
sns.heatmap(
    profile_norm,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    linewidths=0.5
)
plt.title("Normalized Cluster Feature Profile Heatmap")
plt.show()

### data vis 3

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

features = [
    "total_spend", "avg_order_value", "purchase_frequency",
    "recency_days", "avg_discount_percentage", "promo_order_ratio"
]

radar_data = cluster_profile[features].copy()

# 归一化到 0-1，避免量纲差异
radar_norm = (radar_data - radar_data.min()) / (radar_data.max() - radar_data.min())

labels = radar_norm.columns.tolist()
num_vars = len(labels)

angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

for cluster_id, row in radar_norm.iterrows():
    values = row.tolist()
    values += values[:1]
    ax.plot(angles, values, label=f"Cluster {cluster_id}")
    ax.fill(angles, values, alpha=0.2)

ax.set_thetagrids(np.degrees(angles[:-1]), labels)
ax.set_ylim(0, 1)
ax.set_title("Cluster Radar Profile")
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.show()